# **Starting place for, do some imports and define some variables.**


*   Location
*   Staging_Bucket for Agent Engine Deployment
*   Project_ID

For Agent Engine Deployment
You will be prompted for your Google Maps API.  Subesequent runs will not request it.

In [1]:
# 1. Install required packages with the correct extras
!pip install --quiet "google-cloud-aiplatform[agent_engines,adk]" google-adk requests googlemaps

import os
import getpass
import vertexai
from google.colab import userdata

# 2. Initialize global cloud properties
PROJECT_ID = "qwiklabs-gcp-01-d59f380c208c"
LOCATION = "us-central1"
STAGING_BUCKET = "gs://chal5_stage_bucket"

# Explicitly set Quota Project to avoid 'None' value issues
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROJECT_ID

vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET,)
print(f"Vertex AI successfully initialized for project: {PROJECT_ID}")

# 3. Securely prompt for your Google Maps API key
try:
    if not os.environ.get("GOOGLE_MAPS_API_KEY"):
        maps_key = getpass.getpass("Enter your Google Maps API Key (starts with AIzaSy...): ")
        os.environ["GOOGLE_MAPS_API_KEY"] = maps_key
except NameError:
    maps_key = getpass.getpass("Enter your Google Maps API Key (starts with AIzaSy...): ")
    os.environ["GOOGLE_MAPS_API_KEY"] = maps_key

Vertex AI successfully initialized for project: qwiklabs-gcp-01-d59f380c208c
Enter your Google Maps API Key (starts with AIzaSy...): ··········


### **1. Weather Integration Tool**
This section defines the `get_us_weather_by_city_name` tool. It performs a two-step process:
1.  **Geocoding**: Uses the `googlemaps` library to convert a human-readable city name into Latitude and Longitude coordinates.
2.  **NWS API Call**: Uses the coordinates to query the National Weather Service (weather.gov) API to retrieve real-time regional forecasts.

In [2]:
#This is a combined function to get the users lat/long based on a US City and then provide weather for that location.
import os
import requests
import googlemaps
from typing import Any

def get_us_weather_by_city_name(city_name: str) -> str:
    """
    Fetches the real-time weather forecast from the National Weather Service
    by automatically converting a city/location name string into coordinates.
    """
    # 1. Fetch Coordinates from Google Maps
    gmaps_key = os.environ.get("GOOGLE_MAPS_API_KEY")
    gmaps = googlemaps.Client(key=gmaps_key)
    geocode_result = gmaps.geocode(city_name)

    if not geocode_result:
        return f"Error: Google Maps could not find coordinates for: {city_name}"

    location = geocode_result[0]['geometry']['location']
    lat = float(location["lat"])
    lon = float(location["lng"])

    # 2. Fetch Forecast from NWS
    headers = {'User-Agent': 'ADK-Weather-Agent'}
    # Ensure coordinates are formatted correctly in the URL
    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    points_res = requests.get(points_url, headers=headers).json()

    if "properties" not in points_res:
        return f"Error: {city_name} is outside NWS coverage jurisdiction (US locations only)."

    forecast_url = points_res["properties"]["forecast"]
    forecast_res = requests.get(forecast_url, headers=headers).json()
    periods = forecast_res["properties"]["periods"]

    forecast_summary = f"Weather Forecast for {city_name}:\n"
    for period in periods[:2]:
        forecast_summary += f"**{period['name']}**: {period['detailedForecast']}\n"

    return forecast_summary

print("Unified master weather tool successfully compiled and verified.")

Unified master weather tool successfully compiled and verified.


Function used by Map Routing Agent

In [3]:
import googlemaps
from typing import Any
import re
import os

def get_route_directions(origin: str, destination: str) -> str:
    """
    Provides driving directions between two locations using Google Maps.
    Useful for evacuation and finding safe routes.
    """
    # Access the key from the global scope or environment
    gmaps_key = globals().get('maps_key') or os.environ.get("GOOGLE_MAPS_API_KEY")

    if not gmaps_key:
        return "Error: Google Maps API key not found. Please ensure maps_key is defined."

    try:
        gmaps = googlemaps.Client(key=gmaps_key)
        # Simplified request to avoid quota/tier issues
        directions_result = gmaps.directions(
            origin,
            destination,
            mode="driving"
        )

        if not directions_result:
            return f"No routes found between {origin} and {destination}. Check location names."

        route = directions_result[0]['legs'][0]
        summary = f"Safe Route from {origin} to {destination}:\n"
        summary += f"Estimated Distance: {route['distance']['text']}\n"
        summary += f"Estimated Duration: {route['duration']['text']}\n\n"
        summary += "Step-by-step instructions:\n"

        for i, step in enumerate(route['steps'], 1):
            # Clean HTML tags from instructions
            instruction = re.sub('<[^<]+?>', '', step['html_instructions'])
            summary += f'{i}. {instruction} ({step["distance"]["text"]})\n'

        print(f"DEBUG: Successfully retrieved route for {origin} to {destination}")
        return summary
    except Exception as e:
        print(f"DEBUG: Map Tool Error: {str(e)}")
        return f"Google Maps API Error Detail: {str(e)}"

### **2. Security Guardrails & Regulatory Logging**
To ensure safe and transparent AI behavior, we implement **ADK Callbacks**:
*   **Moderation Guardrail**: A `before_model` callback that scans user input for banned keywords (e.g., "hack"). If detected, the agent returns a security notice instead of processing the request.
*   **Audit Logging**: `log_user_prompt` and `log_model_response` callbacks record the conversation flow to the standard output for debugging and compliance.

In [4]:
# Callback functions for logging and content moderation.
import logging
import sys
import warnings
from typing import Optional
from google.adk.models import LlmRequest, LlmResponse

try:
    from google.adk.models import Content, Part
except ImportError:
    Content, Part = None, None

warnings.filterwarnings('ignore', category=UserWarning)

logger = logging.getLogger("ADK-FEMA-Lab")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    logger.addHandler(handler)

def log_user_prompt(callback_context, llm_request: LlmRequest) -> Optional[LlmResponse]:
    try:
        contents = getattr(llm_request, "contents", [])
        if contents:
            last = contents[-1]
            parts = getattr(last, "parts", [])
            if parts:
                text = getattr(parts[0], "text", "")
                logger.info("[%s] logging USER » %s", getattr(callback_context, "agent_name", "Default_Bot"), str(text).strip())
    except Exception as e:
        logger.error(f"Logging error: {e}")
    return None

def moderation_guardrail(callback_context, llm_request: LlmRequest) -> Optional[LlmResponse]:
    banned_keywords = ["hack", "exploit", "override", "bypass"]
    try:
        contents = getattr(llm_request, "contents", [])
        if contents:
            last = contents[-1]
            parts = getattr(last, "parts", [])
            if parts:
                text = str(getattr(parts[0], "text", "")).lower()
                if any(word in text for word in banned_keywords):
                    logger.warning("[%s] MODERATION TRIGGERED", getattr(callback_context, "agent_name", "Default_Bot"))
                    msg = "Security Notice: Banned term detected."
                    if Content and Part:
                        return LlmResponse(content=Content(role="model", parts=[Part(text=msg)]))
                    return LlmResponse(content={"role": "model", "parts": [{"text": msg}]})
    except Exception as e:
        logger.error(f"Guardrail error: {e}")
    return None

def log_model_response(callback_context, llm_response: LlmResponse) -> Optional[LlmResponse]:
    try:
        content = getattr(llm_response, "content", None)
        if content:
            parts = getattr(content, "parts", [])
            for part in parts:
                text = getattr(part, "text", None)
                if text:
                    logger.info("[%s] logging MODEL RESPONSE » %s", getattr(callback_context, "agent_name", "Weather_Bot"), str(text).strip())
    except Exception as e:
        logger.error(f"Response logging error: {e}")
    return None

logger.info("Robust logger and guardrails ready.")

def combined_before_model_handler(callback_context, llm_request: LlmRequest) -> Optional[LlmResponse]:
    guardrail_action = moderation_guardrail(callback_context, llm_request)
    if guardrail_action is not None: return guardrail_action
    return log_user_prompt(callback_context, llm_request)

2026-07-09 18:56:43,174 [INFO] Robust logger and guardrails ready.


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()
INFO:ADK-FEMA-Lab:Robust logger and guardrails ready.


### **3. Agent Orchestration & Native LoopAgent**
This section initializes the specialized agents and the main routing logic:


In [21]:
from google.adk.agents import Agent, LoopAgent
from google.adk.tools import agent_tool, google_search_tool
from vertexai.preview import reasoning_engines

# 1. Specialized Agents
weather_agent = Agent(
    name="Weather_Agent",
    description="US weather forecast tool.",
    instruction="Extract city name. Return a summary of the 2-day forecast.",
    tools=[get_us_weather_by_city_name],
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

google_search_agent = Agent(
    name="Google_Search_Agent",
    description="Emergency alert search tool.",
    instruction="Search ONLY for active emergency alerts, evacuation orders, or road closures.",
    tools=[google_search_tool.GoogleSearchTool()],
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

map_route_agent = Agent(
    name="Map_Route_Agent",
    description="Navigation and routing tool.",
    instruction="Provide only the safest route between the origin and destination, including estimated time, distance, and top 3 maneuvers.",
    tools=[get_route_directions],
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

# 2. Loop Agents (Self-Correcting Units)
critique_agent = Agent(
    name="Critique_Agent",
    description="Verification agent.",
    instruction="Verify if the findings provided are accurate and complete.",
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

refiner_agent = Agent(
    name="Refiner_Agent",
    description="Final response agent.",
    instruction="Synthesize the findings into a professional response.",
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

# Integrating Search directly into the refinement loop
clarity_agent = LoopAgent(
    name="Clarity_Agent",
    description="Emergency Information Retrieval and Refinement Unit.",
    sub_agents=[google_search_agent, critique_agent, refiner_agent],
    max_iterations=1
)

# 3. Root Orchestrator
root_agent = Agent(
    name='Main_Agent',
    description='Disaster Response Orchestrator',
    instruction="""1. For weather, use `Weather_Agent`.
2. For driving routes or navigation, use `Map_Route_Agent`.
3. For any DISASTER, EMERGENCY, or ALERTS, use `Clarity_Agent` (which performs search and verification). If `Clarity_Agent` reports an EVACUATION NOTICE, you MUST explicitly offer to provide a safe route using `Map_Route_Agent`.
4. If the user asks general or off-topic questions, inform them you are specialized for disaster response.
5. Greet the user and explain your capabilities (weather, safe routes, and emergency alerts).""",
    tools=[
        agent_tool.AgentTool(agent=weather_agent),
        agent_tool.AgentTool(agent=map_route_agent),
        agent_tool.AgentTool(agent=clarity_agent)
    ]
)

app = reasoning_engines.AdkApp(agent=root_agent)
print('Architecture optimized: Google_Search_Agent is now integrated into the Clarity_Agent loop.')

Architecture optimized: Google_Search_Agent is now integrated into the Clarity_Agent loop.


Deploy a Agent Platform

In [26]:
import vertexai
from vertexai import agent_engines

# Force local pydantic check and then deploy
# The 400 error usually suggests the Cloud Build cannot satisfy the dependencies or the pickling failed.
remote_agent = agent_engines.create(
    app,
    requirements=[
        "google-cloud-aiplatform[agent_engines,adk]==1.157.0",
        "google-adk==1.29.0",
        "pydantic==2.12.3",
        "cloudpickle==3.1.2",
        "googlemaps",
        "requests"
    ],
        env_vars={
        "GOOGLE_MAPS_API_KEY": maps_key
    },
    display_name="FEMA_v1"
)

print(f"Agent successfully deployed: {remote_agent.resource_name}")

INFO:vertexai.agent_engines:Identified the following requirements: {'google-cloud-aiplatform': '1.157.0', 'pydantic': '2.12.3', 'cloudpickle': '3.1.2'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]==1.157.0', 'google-adk==1.29.0', 'pydantic==2.12.3', 'cloudpickle==3.1.2', 'googlemaps', 'requests']
INFO:vertexai.agent_engines:Using bucket chal5_stage_bucket
INFO:vertexai.agent_engines:Wrote to gs://chal5_stage_bucket/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://chal5_stage_bucket/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://chal5_stage_bucket/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/239267613896/locations/us-central1/reasoningEngines/2492846297598394368/operations/4968361135142076416
INFO:ver

Agent successfully deployed: projects/239267613896/locations/us-central1/reasoningEngines/2492846297598394368


Below code validates the Agent Platform deployed and is working.

In [ ]:
for event in remote_agent.stream_query(
  user_id="agent-engine-test-user",
  message="I am in Enoch, UT and need a safe route to the Cedar City high school.",
):
  print(event)

# **Tests against local ADPApp.  This creates a function called run_agent_test that provides output in a formated way.**

In [16]:
import time
import random
from IPython.display import Markdown, display

def run_agent_test(test_name, message, user_id='test-user', max_retries=3):
    print(f'\n--- Testing: {test_name} ---')
    print(f'Input: "{message}"')

    for attempt in range(max_retries):
        try:
            # Create a fresh session for each test case
            session = app.create_session(user_id=user_id)
            s_id = session['id'] if isinstance(session, dict) else session.id

            response_stream = app.stream_query(user_id=user_id, session_id=s_id, message=message)

            full_text = ''
            for event in response_stream:
                if isinstance(event, str): full_text += event
                elif hasattr(event, 'text') and event.text: full_text += event.text
                elif hasattr(event, 'content'):
                    parts = getattr(event.content, 'parts', []) if not isinstance(event.content, dict) else event.content.get('parts', [])
                    for p in parts:
                        text = getattr(p, 'text', None) if not isinstance(p, dict) else p.get('text')
                        if text: full_text += text
                elif isinstance(event, dict):
                    parts = event.get('content', {}).get('parts', [])
                    for p in parts:
                        if 'text' in p: full_text += p['text']

            if full_text:
                display(Markdown(f'**Response:** {full_text}'))
                return full_text
            else:
                print('System: No text returned.')
                return ''

        except Exception as e:
            if '429' in str(e) and attempt < max_retries - 1:
                wait_time = (2 ** attempt) + random.random()
                print(f'Quota hit (429). Retrying in {wait_time:.2f} seconds...')
                time.sleep(wait_time)
                continue
            else:
                print(f'Error: {e}')
                return None

# **Individual Local Tests, Make sure to run the cell above first to define the run_agent_test function.**

Enoch UT was used because of a Red Flag advisory in place.

In [25]:
run_agent_test('General Emergency', 'I am in Enoch, UT is is there a disaster?')


--- Testing: General Emergency ---
Input: "I am in Enoch, UT is is there a disaster?"
2026-07-09 19:39:48,259 [INFO] [Google_Search_Agent] logging USER » Is there a disaster in Enoch, UT?


INFO:ADK-FEMA-Lab:[Google_Search_Agent] logging USER » Is there a disaster in Enoch, UT?


2026-07-09 19:39:55,781 [INFO] [Google_Search_Agent] logging MODEL RESPONSE » As of Thursday, July 9, 2026, at 7:39 PM UTC, there are no active emergency alerts, evacuation orders, or road closures specifically reported for Enoch, Utah.

While some weather advisories for Enoch appear in search results, they are not currently active. Enoch City Emergency Management partners with Everbridge for emergency notifications, suggesting a system is in place for such events.

Previous disaster events, such as flooding in Enoch in August 2021, were noted in search results, but these are historical and not current incidents. There are also mentions of general statewide road conditions and wildfires in other parts of Utah, but these do not indicate a disaster in Enoch itself.


INFO:ADK-FEMA-Lab:[Google_Search_Agent] logging MODEL RESPONSE » As of Thursday, July 9, 2026, at 7:39 PM UTC, there are no active emergency alerts, evacuation orders, or road closures specifically reported for Enoch, Utah.

While some weather advisories for Enoch appear in search results, they are not currently active. Enoch City Emergency Management partners with Everbridge for emergency notifications, suggesting a system is in place for such events.

Previous disaster events, such as flooding in Enoch in August 2021, were noted in search results, but these are historical and not current incidents. There are also mentions of general statewide road conditions and wildfires in other parts of Utah, but these do not indicate a disaster in Enoch itself.


2026-07-09 19:39:55,783 [INFO] [Google_Search_Agent] logging MODEL RESPONSE » No disaster has been reported in Enoch, UT.


...

INFO:ADK-FEMA-Lab:[Refiner_Agent] logging USER » For context:


2026-07-09 19:40:01,059 [INFO] [Refiner_Agent] logging MODEL RESPONSE » As of Thursday, July 9, 2026, at 7:39 PM UTC, there are no reported active disasters in Enoch, Utah.

Information from current searches indicates:
*   There are no active emergency alerts, evacuation orders, or road closures specifically for Enoch, UT.
*   Any weather advisories found are not currently active.
*   Past incidents, such as flooding in August 2021, are historical and do not represent a current disaster.
*   General statewide road conditions or wildfires in other parts of Utah do not affect Enoch itself.

Enoch City Emergency Management utilizes Everbridge for emergency notifications, and no such notifications are currently active for the area.


INFO:ADK-FEMA-Lab:[Refiner_Agent] logging MODEL RESPONSE » As of Thursday, July 9, 2026, at 7:39 PM UTC, there are no reported active disasters in Enoch, Utah.

Information from current searches indicates:
*   There are no active emergency alerts, evacuation orders, or road closures specifically for Enoch, UT.
*   Any weather advisories found are not currently active.
*   Past incidents, such as flooding in August 2021, are historical and do not represent a current disaster.
*   General statewide road conditions or wildfires in other parts of Utah do not affect Enoch itself.

Enoch City Emergency Management utilizes Everbridge for emergency notifications, and no such notifications are currently active for the area.


**Response:** As of Thursday, July 9, 2026, at 7:39 PM UTC, there are no reported active disasters in Enoch, Utah. There are no active emergency alerts, evacuation orders, or road closures specifically for Enoch, UT, and any weather advisories are not currently active.

'As of Thursday, July 9, 2026, at 7:39 PM UTC, there are no reported active disasters in Enoch, Utah. There are no active emergency alerts, evacuation orders, or road closures specifically for Enoch, UT, and any weather advisories are not currently active.'

In [13]:
run_agent_test('General question', 'I am in Enoch, what is the best ice cream shop in town?')


--- Testing: General question ---
Input: "I am in Enoch, what is the best ice cream shop in town?"


**Response:** I specialize in disaster response, including providing weather updates, safe routes during emergencies, and emergency alerts. I cannot help with finding the best ice cream shop.

'I specialize in disaster response, including providing weather updates, safe routes during emergencies, and emergency alerts. I cannot help with finding the best ice cream shop.'

In [20]:
# Test the new routing capability
run_agent_test('Evacuation Route', 'I am in Enoch, UT and need a safe route to the Cedar City high school.')


--- Testing: Evacuation Route ---
Input: "I am in Enoch, UT and need a safe route to the Cedar City high school."
2026-07-09 19:35:43,256 [INFO] [Map_Route_Agent] logging USER » Route from Enoch, UT to Cedar City high school


INFO:ADK-FEMA-Lab:[Map_Route_Agent] logging USER » Route from Enoch, UT to Cedar City high school


DEBUG: Successfully retrieved route for Enoch, UT to Cedar City high school
2026-07-09 19:35:44,559 [INFO] [Map_Route_Agent] logging USER » None


INFO:ADK-FEMA-Lab:[Map_Route_Agent] logging USER » None


2026-07-09 19:35:45,534 [INFO] [Map_Route_Agent] logging MODEL RESPONSE » A safe route from Enoch, UT to Cedar City high school is as follows:

Estimated Distance: 7.7 miles
Estimated Duration: 14 minutes

Top 3 Maneuvers:
1. Head south on N Enoch Rd toward E 5250 N/Jones Ln.
2. Turn right onto N Old Hwy 91/Old US Hwy 91.
3. Turn left onto UT-130 S.


INFO:ADK-FEMA-Lab:[Map_Route_Agent] logging MODEL RESPONSE » A safe route from Enoch, UT to Cedar City high school is as follows:

Estimated Distance: 7.7 miles
Estimated Duration: 14 minutes

Top 3 Maneuvers:
1. Head south on N Enoch Rd toward E 5250 N/Jones Ln.
2. Turn right onto N Old Hwy 91/Old US Hwy 91.
3. Turn left onto UT-130 S.


**Response:** A safe route from Enoch, UT to Cedar City high school is as follows:

Estimated Distance: 7.7 miles
Estimated Duration: 14 minutes

Top 3 Maneuvers:
1. Head south on N Enoch Rd toward E 5250 N/Jones Ln.
2. Turn right onto N Old Hwy 91/Old US Hwy 91.
3. Turn left onto UT-130 S.

'A safe route from Enoch, UT to Cedar City high school is as follows:\n\nEstimated Distance: 7.7 miles\nEstimated Duration: 14 minutes\n\nTop 3 Maneuvers:\n1. Head south on N Enoch Rd toward E 5250 N/Jones Ln.\n2. Turn right onto N Old Hwy 91/Old US Hwy 91.\n3. Turn left onto UT-130 S.'

In [22]:
run_agent_test('Another Disaster', 'Is there flooding in Dunedin FL?')


--- Testing: Another Disaster ---
Input: "Is there flooding in Dunedin FL?"
2026-07-09 19:37:54,674 [INFO] [Google_Search_Agent] logging USER » Is there flooding in Dunedin FL?


INFO:ADK-FEMA-Lab:[Google_Search_Agent] logging USER » Is there flooding in Dunedin FL?


2026-07-09 19:38:02,133 [INFO] [Google_Search_Agent] logging MODEL RESPONSE » As of Thursday, July 9, 2026, there are no active emergency alerts, evacuation orders, or widespread road closures specifically indicating current flooding in Dunedin, Florida. The Weather Network reports "No Weather Alerts" for Dunedin, FL.

While Dunedin is susceptible to flooding due to various conditions like hurricanes, heavy rain, and storm surge, and a significant percentage of properties have a risk of flooding over the next 30 years, there is no immediate flooding reported.

Past events, such as Hurricane Idalia in August 2023, caused street flooding and water intrusion into approximately 125 homes in Dunedin, leading to warnings against driving through standing water. Pinellas County, which includes Dunedin, utilizes the "Alert Pinellas" system to notify residents of various emergencies, including flooding events and mandatory evacuations. The county also issues evacuation orders based on storm surg

INFO:ADK-FEMA-Lab:[Google_Search_Agent] logging MODEL RESPONSE » As of Thursday, July 9, 2026, there are no active emergency alerts, evacuation orders, or widespread road closures specifically indicating current flooding in Dunedin, Florida. The Weather Network reports "No Weather Alerts" for Dunedin, FL.

While Dunedin is susceptible to flooding due to various conditions like hurricanes, heavy rain, and storm surge, and a significant percentage of properties have a risk of flooding over the next 30 years, there is no immediate flooding reported.

Past events, such as Hurricane Idalia in August 2023, caused street flooding and water intrusion into approximately 125 homes in Dunedin, leading to warnings against driving through standing water. Pinellas County, which includes Dunedin, utilizes the "Alert Pinellas" system to notify residents of various emergencies, including flooding events and mandatory evacuations. The county also issues evacuation orders based on storm surge risk, as se

2026-07-09 19:38:02,141 [INFO] [Critique_Agent] logging USER » For context:


...

INFO:ADK-FEMA-Lab:[Refiner_Agent] logging USER » For context:


2026-07-09 19:38:15,884 [INFO] [Refiner_Agent] logging MODEL RESPONSE » As of Thursday, July 9, 2026, there are no active emergency alerts, evacuation orders, or widespread road closures specifically indicating current flooding in Dunedin, Florida. The Weather Network also reports "No Weather Alerts" for the area.

While Dunedin is generally susceptible to flooding due to factors like hurricanes, heavy rainfall, and storm surge, and a significant percentage of properties have a long-term flood risk, there is no immediate flooding reported. Past events, such as Hurricane Idalia in August 2023, did cause street flooding and home intrusion. Current road closures in the area are related to specific events or construction, not general flooding.

Residents are encouraged to utilize resources such as the "Alert Pinellas" system and the "Ready Pinellas" app to stay informed about potential emergencies, including any future flooding events or evacuation orders.


INFO:ADK-FEMA-Lab:[Refiner_Agent] logging MODEL RESPONSE » As of Thursday, July 9, 2026, there are no active emergency alerts, evacuation orders, or widespread road closures specifically indicating current flooding in Dunedin, Florida. The Weather Network also reports "No Weather Alerts" for the area.

While Dunedin is generally susceptible to flooding due to factors like hurricanes, heavy rainfall, and storm surge, and a significant percentage of properties have a long-term flood risk, there is no immediate flooding reported. Past events, such as Hurricane Idalia in August 2023, did cause street flooding and home intrusion. Current road closures in the area are related to specific events or construction, not general flooding.

Residents are encouraged to utilize resources such as the "Alert Pinellas" system and the "Ready Pinellas" app to stay informed about potential emergencies, including any future flooding events or evacuation orders.


**Response:** As of Thursday, July 9, 2026, there is no current flooding reported in Dunedin, Florida. There are no active emergency alerts, evacuation orders, or widespread road closures. The Weather Network also reports "No Weather Alerts" for the area.

'As of Thursday, July 9, 2026, there is no current flooding reported in Dunedin, Florida. There are no active emergency alerts, evacuation orders, or widespread road closures. The Weather Network also reports "No Weather Alerts" for the area.'

In [24]:
run_agent_test('Real Disaster', 'I am in Vantage, Washington. Should I evacuate?')


--- Testing: Real Disaster ---
Input: "I am in Vantage, Washington. Should I evacuate?"
2026-07-09 19:38:56,485 [INFO] [Google_Search_Agent] logging USER » Evacuation notice for Vantage, Washington


INFO:ADK-FEMA-Lab:[Google_Search_Agent] logging USER » Evacuation notice for Vantage, Washington


2026-07-09 19:39:00,744 [INFO] [Google_Search_Agent] logging MODEL RESPONSE » As of Thursday, July 9, 2026, evacuation orders for Vantage, Washington, due to a wildfire have been downgraded from Level 3 "Go Now" to Level 2 "Be Ready".

A rapidly moving wildfire prompted the initial Level 3 "Go Now" evacuation orders on Wednesday evening, July 8, 2026, for the city of Vantage. Interstate 90 was also impacted, with closures in both directions. While eastbound I-90 has since reopened, the Vantage Highway remains closed at Parke Creek Road. At least one structure was reported to have burned, and there were downed powerlines. The cause of the fire, which started around 5:30 p.m. on Wednesday, has not yet been determined.


INFO:ADK-FEMA-Lab:[Google_Search_Agent] logging MODEL RESPONSE » As of Thursday, July 9, 2026, evacuation orders for Vantage, Washington, due to a wildfire have been downgraded from Level 3 "Go Now" to Level 2 "Be Ready".

A rapidly moving wildfire prompted the initial Level 3 "Go Now" evacuation orders on Wednesday evening, July 8, 2026, for the city of Vantage. Interstate 90 was also impacted, with closures in both directions. While eastbound I-90 has since reopened, the Vantage Highway remains closed at Parke Creek Road. At least one structure was reported to have burned, and there were downed powerlines. The cause of the fire, which started around 5:30 p.m. on Wednesday, has not yet been determined.


2026-07-09 19:39:00,751 [INFO] [Critique_Agent] logging USER » For context:


...

INFO:ADK-FEMA-Lab:[Refiner_Agent] logging USER » For context:


2026-07-09 19:39:21,136 [INFO] [Refiner_Agent] logging MODEL RESPONSE » **Subject: Update: Wildfire Evacuation Notice for Vantage, Washington**

**Key Developments:**

As of Thursday, July 9, 2026, the evacuation orders for Vantage, Washington, have been **downgraded from Level 3 "Go Now" to Level 2 "Be Ready"**.

**Incident Overview:**

A rapidly moving wildfire prompted initial Level 3 "Go Now" evacuation orders for the city of Vantage on Wednesday evening, July 8, 2026, starting around 5:30 p.m. The cause of the fire is currently undetermined.

**Impacts:**

*   **Evacuation Status:** While residents are no longer under immediate mandatory evacuation, those in Vantage should be prepared to evacuate at a moment's notice.
*   **Road Closures:**
    *   Interstate 90 was initially closed in both directions but the eastbound lanes have since reopened.
    *   Vantage Highway remains closed at Parke Creek Road.
*   **Reported Damage:** At least one structure has been reported burned, and

INFO:ADK-FEMA-Lab:[Refiner_Agent] logging MODEL RESPONSE » **Subject: Update: Wildfire Evacuation Notice for Vantage, Washington**

**Key Developments:**

As of Thursday, July 9, 2026, the evacuation orders for Vantage, Washington, have been **downgraded from Level 3 "Go Now" to Level 2 "Be Ready"**.

**Incident Overview:**

A rapidly moving wildfire prompted initial Level 3 "Go Now" evacuation orders for the city of Vantage on Wednesday evening, July 8, 2026, starting around 5:30 p.m. The cause of the fire is currently undetermined.

**Impacts:**

*   **Evacuation Status:** While residents are no longer under immediate mandatory evacuation, those in Vantage should be prepared to evacuate at a moment's notice.
*   **Road Closures:**
    *   Interstate 90 was initially closed in both directions but the eastbound lanes have since reopened.
    *   Vantage Highway remains closed at Parke Creek Road.
*   **Reported Damage:** At least one structure has been reported burned, and downed power

**Response:** As of Thursday, July 9, 2026, the evacuation orders for Vantage, Washington, have been **downgraded from Level 3 "Go Now" to Level 2 "Be Ready"**. This means you are no longer under immediate mandatory evacuation, but you should be prepared to evacuate at a moment's notice.

Please note that Interstate 90 eastbound has reopened, but Vantage Highway remains closed at Parke Creek Road.

Would you like me to help you find a safe route, perhaps for a preparedness plan?

'As of Thursday, July 9, 2026, the evacuation orders for Vantage, Washington, have been **downgraded from Level 3 "Go Now" to Level 2 "Be Ready"**. This means you are no longer under immediate mandatory evacuation, but you should be prepared to evacuate at a moment\'s notice.\n\nPlease note that Interstate 90 eastbound has reopened, but Vantage Highway remains closed at Parke Creek Road.\n\nWould you like me to help you find a safe route, perhaps for a preparedness plan?'